# Sunuwar TTS — Phase 6 prototype fine-tune (Colab T4)

Fine-tunes `facebook/mms-tts-nep` (VITS) on the Sunuwar single-speaker dataset
built by `src/build_tts_dataset.py`.

**Set expectations before you run this.** The current dataset is **0.88 hours**
(472 clips) from a 12-chapter alignment sample. That is enough to prove the
pipeline end-to-end and to produce recognisably Sunuwar-sounding speech; it is
*not* enough for an intelligible voice. Single-speaker VITS fine-tunes usually
want 5–20h. The full corpus is 29.5h and will get there once Phase 3 alignment
has run on all 260 chapters.

What this notebook is really for: burning down the integration risks (tokenizer
coverage, discriminator conversion, data contract, checkpointing) while the data
is small and iteration is cheap.

**Runtime → Change runtime type → T4 GPU** before starting.

## 1. Environment check

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 2. Mount Drive

Checkpoints go to Drive so a session timeout costs minutes, not the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/sunuwar_tts', exist_ok=True)

## 3. Get the dataset into Colab

On your laptop, zip the dataset (97 MB) and put it in Drive at
`MyDrive/sunuwar_tts/tts_dataset.zip`:

```powershell
Compress-Archive -Path D:\Lost-Voices\data\processed\tts_dataset\* `
                 -DestinationPath D:\Lost-Voices\tts_dataset.zip
```

The audio is licensed CC BY-NC-ND and must not be redistributed, so keep it in
your own Drive — do not push it to a public Hub dataset.

In [ ]:
!rm -rf /content/tts_dataset
!mkdir -p /content/tts_dataset
!unzip -q /content/drive/MyDrive/sunuwar_tts/tts_dataset.zip -d /content/tts_dataset

import csv, glob
for split in ('train', 'validation'):
    rows = list(csv.DictReader(open(f'/content/tts_dataset/{split}/metadata.csv', encoding='utf-8')))
    wavs = glob.glob(f'/content/tts_dataset/{split}/*.wav')
    print(f'{split:<11} {len(rows):>4} metadata rows, {len(wavs):>4} wavs')
    assert len(rows) == len(wavs), 'metadata/wav count mismatch — re-zip the dataset'

## 4. Get the project code

Set `REPO_URL` to your GitHub remote. If the repo is private, use a token URL
(`https://<token>@github.com/<user>/<repo>.git`) or just upload `src/train_tts.py`
and `configs/tts.yaml` by hand into `/content/Lost-Voices/`.

In [ ]:
REPO_URL = ''  # <-- fill this in

import os
if REPO_URL and not os.path.isdir('/content/Lost-Voices'):
    !git clone $REPO_URL /content/Lost-Voices

assert os.path.isfile('/content/Lost-Voices/src/train_tts.py'), \
    'train_tts.py not found — clone the repo or upload src/ and configs/ manually'
print('project code ready')

## 5. Install the VITS training implementation

Mainline `transformers` ships `VitsModel` for **inference only** — the released
checkpoints have no discriminator and the class has no training loss. VITS is a
GAN, so training needs the discriminator, the mel/KL losses and monotonic
alignment search. `ylacombe/finetune-hf-vits` is the reference implementation
that supplies them.

The `monotonic_align` Cython extension must be compiled — training will fail at
the first step without it.

In [ ]:
%cd /content
![ -d finetune-hf-vits ] || git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd /content/finetune-hf-vits
!pip install -q -r requirements.txt

# Build the Cython monotonic alignment extension
%cd /content/finetune-hf-vits/monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd /content/finetune-hf-vits

In [ ]:
# Verify the pieces that silently break training if missing
import importlib, subprocess

for mod in ('torch', 'transformers', 'datasets', 'accelerate', 'soundfile', 'librosa'):
    m = importlib.import_module(mod)
    print(f'{mod:<14} {getattr(m, "__version__", "?")}')

import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

try:
    import monotonic_align
    print('monotonic_align: OK')
except Exception as e:
    print('monotonic_align FAILED:', e)

## 6. Convert the base checkpoint so it carries a discriminator

`facebook/mms-tts-nep` as published is generator-only. This pulls the original
MMS release for Nepali and writes a local checkpoint with the discriminator
weights attached, which is what we actually fine-tune from.

In [ ]:
%cd /content/finetune-hf-vits
!python convert_original_discriminator_checkpoint.py \
    --language_code nep \
    --pytorch_dump_folder_path /content/mms-tts-nep-train

!ls -la /content/mms-tts-nep-train

## 7. Preflight: does the Nepali tokenizer cover Sunuwar text?

**This is the step most likely to matter.** MMS tokenizers are character-based,
so any character the checkpoint has never seen becomes `<unk>` — silently, with
no error. Sunuwar text is ~3.5% ZWJ (U+200D) by character count.

Read the report before continuing. If ZWJ is missing, `train_tts.py` drops it,
which is phonemically safe: a ZWJ after a virama (U+094D) only picks the
half-form glyph over the ligature, and the virama that actually encodes the
conjunct is kept. It is a rendering distinction, not a pronunciation one.

In [ ]:
%cd /content/Lost-Voices
!python src/train_tts.py configs/tts.yaml --preflight

## 8. Check our config against the trainer's actual argument names

`run_vits_finetuning.py` changes over time. Rather than discovering a renamed
field after a long download, list what it accepts and diff our keys against it.
Unknown keys are usually harmless (they get ignored) but a *renamed* key means
our setting is silently not applied — check anything flagged here.

In [ ]:
import re, json, sys
sys.path.insert(0, '/content/Lost-Voices/src')
import yaml
from pathlib import Path
from train_tts import build_training_json

cfg = yaml.safe_load(Path('/content/Lost-Voices/configs/tts.yaml').read_text(encoding='utf-8'))
ours = build_training_json(cfg, Path('/content/tts_dataset'), Path('/content/_tts_finetune.json'))

src = Path('/content/finetune-hf-vits/run_vits_finetuning.py').read_text(encoding='utf-8')
known = set(re.findall(r'^\s{4}(\w+)\s*:\s*\w', src, flags=re.M))
known |= set(re.findall(r'--([\w-]+)', src))

unknown = sorted(k for k in ours if k not in known)
print(f'{len(ours)} keys in our config, {len(known)} field names found in the trainer')
print('NOT FOUND in trainer source:', unknown if unknown else 'none')
print()
print(json.dumps(ours, indent=2)[:1500])

### If `dataset_name`/`data_dir` isn't supported

Some revisions of the trainer only accept a Hub dataset id. If the cell above
flags `data_dir`, run this to materialise the audiofolder dataset to disk and
point the trainer at that instead. Keeps the audio local — no upload.

In [ ]:
# Only needed if the compatibility check flagged data_dir.
from datasets import load_dataset, Audio

ds = load_dataset('audiofolder', data_dir='/content/tts_dataset')
ds = ds.cast_column('audio', Audio(sampling_rate=16000))
print(ds)
print(ds['train'][0]['text'])
ds.save_to_disk('/content/tts_dataset_hf')

## 9. Train

~45 steps/epoch at batch 8. 200 epochs ≈ 9k steps, roughly 1.5–2.5h on a T4.
Checkpoints land in Drive every 250 steps, so a disconnect is recoverable —
resume by pointing `model_name_or_path` at the last checkpoint.

Watch the mel loss. If it plateaus early and the audio stays buzzy, that is the
0.88h dataset talking, not a bug.

In [ ]:
%cd /content/Lost-Voices
!python src/train_tts.py configs/tts.yaml

## 10. Synthesise — the bit to show your supervisor

Generates speech from held-out **validation** sentences (chapters LUK_005 and
REV_021, which the model never saw) and plays the real recording beside the
synthesised one for comparison.

In [ ]:
import csv, glob, torch, scipy.io.wavfile
from pathlib import Path
from IPython.display import Audio as PlayAudio, display
from transformers import VitsModel, AutoTokenizer

CKPT = sorted(glob.glob('/content/drive/MyDrive/sunuwar_tts/prototype/checkpoint-*'),
              key=lambda p: int(p.rsplit('-', 1)[1]))
CKPT = CKPT[-1] if CKPT else '/content/drive/MyDrive/sunuwar_tts/prototype'
print('loading', CKPT)

model = VitsModel.from_pretrained(CKPT).eval()
tokenizer = AutoTokenizer.from_pretrained(CKPT)

rows = list(csv.DictReader(open('/content/tts_dataset/validation/metadata.csv', encoding='utf-8')))
out_dir = Path('/content/drive/MyDrive/sunuwar_tts/samples'); out_dir.mkdir(parents=True, exist_ok=True)

for row in rows[:5]:
    inputs = tokenizer(row['text'], return_tensors='pt')
    with torch.no_grad():
        wav = model(**inputs).waveform[0].cpu().numpy()

    name = Path(row['file_name']).stem
    scipy.io.wavfile.write(out_dir / f'{name}_synth.wav', model.config.sampling_rate, wav)

    print('=' * 80)
    print(row['text'])
    print('reference:');   display(PlayAudio(f"/content/tts_dataset/validation/{row['file_name']}"))
    print('synthesised:'); display(PlayAudio(wav, rate=model.config.sampling_rate))

## 11. Free-text synthesis

Type any Sunuwar sentence. Remember the same text adaptation the training data
went through (e.g. ZWJ removal) has to be applied here too, or the tokenizer
sees characters the model never trained on.

In [ ]:
import sys; sys.path.insert(0, '/content/Lost-Voices/src')
from train_tts import preflight, build_policy, adapt_text

TEXT = 'परमप्रभु यावे आ दाक्शो पा'   # <-- your sentence here

missing = {c: 1 for c in set(TEXT) if c not in set(tokenizer.get_vocab()) and not c.isspace()}
clean = adapt_text(TEXT, missing, build_policy(missing, cfg)) if missing else TEXT
if missing:
    print('adapted:', [f'U+{ord(c):04X}' for c in missing], '->', clean)

inputs = tokenizer(clean, return_tensors='pt')
with torch.no_grad():
    wav = model(**inputs).waveform[0].cpu().numpy()
display(PlayAudio(wav, rate=model.config.sampling_rate))

## What to report back

Bring these numbers to the next session — they decide the Phase 3 settings:

1. **Preflight output (cell 7)** — which characters the Nepali vocab lacked, and
   what fraction of text they covered. Determines whether the tokenizer needs
   extending rather than the text being adapted.
2. **Final train/eval loss** and whether it was still descending at the end.
3. **How the samples sound** — specifically whether it is Sunuwar-*sounding*
   nonsense (prosody transferred, phonetics not yet learned — expected at
   0.88h) or pure noise (a real bug).
4. **Wall-clock time for 200 epochs** — scales the full-corpus estimate.